# Stage 3 - Validate and inventory FIRMS data

**NASA Space Apps 2026 - MODIS/VIIRS hotspot harmonization**

Run the five-pass data gate over the staged FIRMS MODIS/VIIRS CSV archives and inspect the audit report. The gate is fail-closed: structural/semantic problems quarantine rows or abort with machine-readable reasons; quarantine is never silent and missing observations are never zero activity.

| | |
|---|---|
| Stage | 3 of 6 - offline compute |
| Network required | NO |
| Runs | the committed CLI scripts (`scripts/`) - logic stays out of cells |
| Writes to | `out/gate/ (inventory + audit reports)` |

Rules that make this notebook reproducible (docs/DEVELOPMENT.md): record the
environment below, keep every output under the working directory, never fetch
from PyPI/GitHub/Hugging Face/REST APIs during offline compute, and keep
generated weights/datasets out of Git.


## 0. Environment and paths

`SOURCE_DIR` points at the read-only FIRMS archive (Kaggle input dataset or a local directory). Files are attributed to MODIS/VIIRS by filename.

In [ ]:
import sys, platform, subprocess
from pathlib import Path

# Kaggle working dir is the ONLY writable location during offline compute.
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "out"
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()

print("python   :", sys.version)
print("platform :", platform.platform())
print("machine  :", platform.machine())
print("repo     :", REPO_ROOT)
print("work dir :", WORK_DIR)

RECORD = {"python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine()}


SOURCE_DIR = Path("/kaggle/input") / "firms"   # adjust to the mounted dataset
if not SOURCE_DIR.exists():
    SOURCE_DIR = REPO_ROOT / "tests" / "fixtures" / "sample_aoi"  # synthetic demo
print("source dir:", SOURCE_DIR)
files = sorted(SOURCE_DIR.glob("*.csv"))
print(f"{len(files)} CSV files:", ", ".join(p.name for p in files))


## 1. Run the five-pass gate

Passes: structural integrity, semantic row quarantine, leakage/split boundaries, transformation invariants, and the machine-readable report (docs/MODEL_CONTRACTS.md section 2).

In [ ]:
import json, hashlib
from pathlib import Path

def run_cli(script: str, *args: str, expect: int = 0) -> subprocess.CompletedProcess:
    """Run a committed CLI script; fail the notebook loudly on bad exits."""
    cmd = [sys.executable, str(REPO_ROOT / "scripts" / script), *args]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200, cwd=str(REPO_ROOT))
    if proc.returncode != expect:
        print(proc.stdout[-4000:]); print(proc.stderr[-4000:])
        raise RuntimeError(f"{script} exited {proc.returncode}, expected {expect}")
    print(proc.stdout[-2000:])
    return proc

def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def latest_artifact(base: Path, prefix: str) -> Path:
    dirs = sorted((p for p in base.glob(prefix + "_*") if p.is_dir()), key=lambda p: p.stat().st_mtime)
    if not dirs:
        raise FileNotFoundError(f"no {prefix}_* artifact under {base}; did the training step run?")
    return dirs[-1]


gate_out = WORK_DIR / "gate"
proc = run_cli("validate_data.py", "--source-dir", str(SOURCE_DIR), "--output-dir", str(gate_out))


## 2. Inspect the audit report

Row counts, quarantined rows with reason codes, split coverage, and hashes are all recorded; extreme FRP values stay in the data as review flags (they are not proof of bad data).

In [ ]:
import json
from pathlib import Path

report = json.loads((gate_out / "dataset_audit_report.json").read_text())
inventory = json.loads((gate_out / "inventory_manifest.json").read_text())

print("== files ==")
for entry in inventory.get("files", []):
    print(f"  {entry.get('path', entry.get('file', '?'))}: {entry.get('rows', '?')} rows, sha256={str(entry.get('sha256', '?'))[:12]}...")

def show(section, obj):
    if isinstance(obj, dict):
        print(f"== {section} ==")
        for key, value in list(obj.items())[:12]:
            print(f"  {key}: {str(value)[:100]}")

for section, payload in report.items():
    show(section, payload)
print("gate status:", report.get("status", report.get("passed")))


## 3. Fail-closed checkpoint

If the gate reports any critical error, **stop here** - training notebooks call `assert_gate_passed` again, but fixing data first keeps every later artifact trustworthy.